# 50. Alert policies built on spectral timing and HGB score plateaus (BTC spot)

I ask whether spectral information can improve when a fixed-score HGB warning fires, once I handle tree-score ties and the first-come-first-served alert scheduler explicitly. Data: BTCUSDT spot, with 2021 training, threshold calibration on 2022 without outcomes, and exploratory evaluation on 2023 to April 2026.

- First I reconstruct the notebook 47 HGB Base and HGB+Spectral models exactly, including 9/16 onset-anchored and 13/12 breach-anchored catches and 800 alerts per model.
- I fixed eight policy arms before seeing any new results, and they use only present and past indicators: `BASE` (notebook 47 HGB without spectral), `HS_ORIGINAL` (notebook 47 HGB+Spectral), `HS_STRICT` (cutoff ties excluded), `HS_TIE_LEVEL` (cutoff ties kept only for a high spectral signal), `HS_TIE_RISE15` (cutoff ties kept only when the spectral signal has risen over 15 minutes), `HS_JITTER_LEVEL` (continuous score perturbation from the spectral level), `HS_HALFMONTH` (monthly budget split 10+10) and `HS_JITTER_HALFMONTH` (jitter combined with 10+10). I tune nothing and select no winner on 2023–2026 outcomes.
- All policies obey the frozen limits of 20 alerts per month, 240 per year and a 240-minute cooldown; half-month policies also cap each half at 10 alerts, and I record realized alert counts.
- Endpoints: the 4-hour breach-anchored pre-onset catches are primary; 4-hour onset-anchored catches are a post hoc secondary endpoint, and I report the 24-hour definition both ways.
- Status: exploratory and limited to the development period; since the primary notebook 47 screen failed, no result here is a confirmation, and I claim no independent validation. I record the policy specification in `50_SPEC.json`.
- Results folder: `spectral_tie_timing_policy_development_v1`.

In [1]:
from pathlib import Path
import sys, subprocess, json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

try:
    import pyarrow
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])

try:
    from numba import njit
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab. Configure ROOT manually to run with local files.')

ROOT = Path('/content/drive/MyDrive/btc_lead_cost_results_77months_spot')
CACHE = ROOT/'cache'
REPRO47 = ROOT/'spectral_nonlinear_interactions_v1'
OUT = ROOT/'spectral_tie_timing_policy_development_v1'
OUT.mkdir(parents=True, exist_ok=True)
NS = 60*10**9
T0 = pd.Timestamp('2021-01-01', tz='UTC').value
T1 = pd.Timestamp('2022-01-01', tz='UTC').value
T2 = pd.Timestamp('2023-01-01', tz='UTC').value
E0 = pd.Timestamp('2023-01-01', tz='UTC').value
E1 = pd.Timestamp('2026-05-01', tz='UTC').value
LEAD_MIN, LEAD_MAX = 30, 240
COOLDOWN = 240
BUDGET_MONTH, BUDGET_YEAR = 20, 240
SPEC = {
    'notebook': '47', 'status': 'exploratory_posthoc',
    'source': 'BTC spot frozen cache', 'train': '2021',
    'target': '4% breach 30-240 min ahead',
    'threshold_calibration': '2022_unlabeled',
    'evaluation': '2023-01 through 2026-04',
    'models': ['Base_Logit','HS_Linear','HS_Interactions','HGB_Base','HGB_HS'],
    'interaction_features': ['z_hs*z_vol','z_hs*(tsr-0.5)','z_hs^2'],
    'clip': 6.0,
    'HGB': dict(max_iter=100, max_leaf_nodes=7, learning_rate=0.05,
                min_samples_leaf=100, l2_regularization=5.0, max_bins=127,
                early_stopping=False, random_state=20261147,
                class_weight='balanced_via_sample_weight'),
    'comparisons': [('HS_Linear','HS_Interactions'),('HGB_Base','HGB_HS')],
    'lead_minutes': [30,240], 'monthly_cap': 20,
    'annual_cap': 240, 'cooldown_minutes': 240,
    'primary': '4h breach-anchored pre-onset catches per 100 alerts',
    'secondary': '4h within-RV-decile pre-onset AUC; onset-anchor and H=1440 descriptive',
    'stop_go': {'within_rv_decile_auc_gain':0.02,
                'pre_catches_per_100_alerts_gain':0.5},
    'selection_note': 'Do not choose a winner from these repeatedly inspected development outcomes',
    'no_holdout_or_prospective_inputs': True
}
(OUT/'50_reproduction_SPEC.json').write_text(json.dumps(SPEC,indent=2))
print('Output:',OUT)

Mounted at /content/drive
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/spectral_tie_timing_policy_development_v1


## 1. BTC spot inputs and event catalog

I reuse the notebook 47–49 reconstruction, with `z_hs` from existing non-anticipating features, and reproduce the development event catalog of 83 events at H=240 and 218 at H=1440.

In [2]:
for p in [CACHE/'BTCUSDT_1m_regularized.parquet', CACHE/'btc_signals_1m.parquet', CACHE/'DATA_SOURCE.json']:
    if not p.is_file():
        raise FileNotFoundError(f'Required frozen spot cache missing: {p}')
source = json.loads((CACHE/'DATA_SOURCE.json').read_text())
assert source.get('venue')=='spot', 'The cache must be BTC SPOT; no futures substitute.'
px = pd.read_parquet(CACHE/'BTCUSDT_1m_regularized.parquet').sort_index()
sg = pd.read_parquet(CACHE/'btc_signals_1m.parquet', columns=['z_vol','tsr','z_hs']).sort_index()
for d in (px, sg):
    assert isinstance(d.index, pd.DatetimeIndex) and d.index.tz is not None
    d.index = d.index.tz_convert('UTC')
assert px.index.equals(sg.index), 'Price/signal minute indexes differ'
T = px.index.asi8
assert np.all(np.diff(T)==NS), 'Index is not a complete one-minute grid'
close = px.close.to_numpy(dtype=float)
volume = px.volume.to_numpy(dtype=float)
X = sg[['z_vol','tsr','z_hs']].to_numpy(dtype=float)
valid = (np.isfinite(close)&(close>0)&np.isfinite(volume)&(volume>0)&
         np.isfinite(X).all(axis=1)&(X[:,1]>=0)&(X[:,1]<=1))
print('rows',len(T),'train valid',int((valid&(T>=T0)&(T<T1)).sum()),'eval valid',int((valid&(T>=E0)&(T<E1)).sum()))

def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory: continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi],'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    if not rows: raise RuntimeError('Event catalog empty')
    return pd.DataFrame(rows)

CAT = {h:price_only_catalog(close,volume,px.index,H=h) for h in (240,1440)}
EVAL = {h:CAT[h].loc[(CAT[h].breach_utc.astype('int64')>=E0+LEAD_MAX*NS)&
                             (CAT[h].breach_utc.astype('int64')<E1)].copy().reset_index(drop=True)
        for h in (240,1440)}
# Onset-anchored evaluation needs a complete onset-240 ... onset-30 window.
EVAL_ON = {h: EVAL[h].loc[EVAL[h].onset_1pct_utc.astype('int64') >= E0 + LEAD_MAX*NS].reset_index(drop=True)
           for h in (240,1440)}
# Mandatory guard against the reference 4-hour development event count
assert len(EVAL[240])==83, f'Expected 83 development events at H=240, found {len(EVAL[240])}'
assert len(EVAL[1440])==218, f'Expected 218 events at H=1440, found {len(EVAL[1440])}'
# Optional: exact timestamp comparison with notebook 2
ref = ROOT/'event_definition_eligibility_audit'/'price_only_event_catalog.csv'
if ref.is_file():
    q = pd.read_csv(ref)
    for c in ('peak_utc','onset_1pct_utc','breach_utc','confirmed_utc'):
        a = pd.to_datetime(q[c],utc=True).astype('int64').to_numpy()
        b = CAT[240][c].astype('int64').to_numpy()
        assert np.array_equal(a,b), f'Notebook 2 mismatch for {c}'
    print('PASS: notebook-2 event timestamps reproduced.')

# Labels from the next qualifying event timestamp (same searchsorted convention as notebook 41).
def future_label(T, events_ns):
    z=np.sort(np.asarray(events_ns,dtype=np.int64))
    positions=np.searchsorted(z,T+LEAD_MIN*NS,side='left')
    y=np.zeros(len(T),dtype=np.int8)
    good=positions<len(z)
    y[good]=(z[positions[good]]<=T[good]+LEAD_MAX*NS).astype(np.int8)
    return y

cat4=CAT[240]
y_breach = future_label(T,cat4.breach_utc.astype('int64').to_numpy())
y_onset = future_label(T,cat4.onset_1pct_utc.astype('int64').to_numpy())
# No retrospective classification rule is used as a runtime signal. During onset training,
# remove confirmed-event in-progress minutes from the negative training examples.
in_progress = np.zeros(len(T),bool)
for e in cat4.itertuples(index=False):
    a=np.searchsorted(T,e.onset_1pct_utc.value,'left')
    b=np.searchsorted(T,e.confirmed_utc.value,'right')
    in_progress[a:b]=True
TRAIN = valid &(T>=T0)&(T<T1-(LEAD_MAX+5)*NS)
TRAIN_ONSET = TRAIN & ~in_progress
CAL = valid &(T>=T1)&(T<T2)
DEV = valid &(T>=E0)&(T<E1-(LEAD_MAX+5)*NS)
print('2021 breach positive minutes:',int(y_breach[TRAIN].sum()))
print('2021 onset positive minutes:',int(y_onset[TRAIN_ONSET].sum()))
if not (y_breach[TRAIN].sum()>50 and y_onset[TRAIN_ONSET].sum()>50):
    raise RuntimeError('Insufficient training positives: inspect event labels and cache.')

rows 3329280 train valid 524332 eval valid 1750857
PASS: notebook-2 event timestamps reproduced.
2021 breach positive minutes: 18779
2021 onset positive minutes: 18771


## 2. The 2021 HGB models of notebook 47, refit unchanged

I reuse the notebook 47 HGB hyperparameters, predictors, balanced sample weights and unlabeled 2022 cutoffs, and I refit the other regression arms too, so that the code path is identical to notebook 47.

In [3]:
# Five arms. All use the same breach target and TRAIN mask.
# This isolates the predictor form, unlike comparisons that also change the target.
CLIP=6.0
zv=np.clip(X[:,0],-CLIP,CLIP)
ts=X[:,1]
hs=np.clip(X[:,2],-CLIP,CLIP)
# Fit the baseline M0 and the linear z_hs ablation on the untouched raw predictors.
X_INTERACT=np.column_stack([X[:,0],X[:,1],X[:,2],hs*zv,hs*(ts-.5),hs*hs])
assert np.isfinite(X_INTERACT[valid]).all(), 'Feature construction produced non-finite observations'

LOGIT_BASE=make_pipeline(StandardScaler(),LogisticRegression(
    C=0.2,class_weight='balanced',max_iter=400,solver='lbfgs',random_state=17))
LOGIT_HS=make_pipeline(StandardScaler(),LogisticRegression(
    C=0.2,class_weight='balanced',max_iter=400,solver='lbfgs',random_state=17))
LOGIT_INTER=make_pipeline(StandardScaler(),LogisticRegression(
    C=0.2,class_weight='balanced',max_iter=400,solver='lbfgs',random_state=17))
LOGIT_BASE.fit(X[TRAIN][:,:2],y_breach[TRAIN])
LOGIT_HS.fit(X[TRAIN],y_breach[TRAIN])
LOGIT_INTER.fit(X_INTERACT[TRAIN],y_breach[TRAIN])

# Keep baseline guard from notebook 46 / notebook 16 if present.
frozen=ROOT/'methods_definitions_v1'/'METHODS_FROZEN.json'
if frozen.is_file():
    f=json.loads(frozen.read_text())
    cfg=f['configs']['M0_VF_logit|4pct']
    assert np.allclose(LOGIT_BASE[-1].coef_.ravel(),cfg['coef'],rtol=1e-6,atol=1e-9), 'M0 coefficient mismatch'
    assert np.isclose(LOGIT_BASE[-1].intercept_[0],cfg['intercept'],rtol=1e-6,atol=1e-9), 'M0 intercept mismatch'
    print('PASS: frozen M0 logistic parameters')
else:
    print('Warning: frozen M0 file missing; the AUC and alert guards still apply')

# Small nonlinear model: exact same hyperparameters with vs without spectral input.
# HGB supports sample_weight in all relevant sklearn releases; avoid version-dependent class_weight.
_,cnt=np.unique(y_breach[TRAIN],return_counts=True)
n_pos=int(y_breach[TRAIN].sum());n_neg=int(TRAIN.sum()-n_pos)
assert n_pos>50 and n_neg>n_pos
train_w=np.where(y_breach[TRAIN]==1,(n_pos+n_neg)/(2*n_pos),(n_pos+n_neg)/(2*n_neg))
def fixed_hgb():
    return HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=7,
        learning_rate=0.05,min_samples_leaf=100,l2_regularization=5.0,
        max_bins=127,early_stopping=False,random_state=20261147)
HGB_BASE=fixed_hgb();HGB_HS=fixed_hgb()
HGB_BASE.fit(X[TRAIN][:,:2],y_breach[TRAIN],sample_weight=train_w)
HGB_HS.fit(X[TRAIN],y_breach[TRAIN],sample_weight=train_w)
MODELS={
 'Base_Logit':(LOGIT_BASE,'base'),
 'HS_Linear':(LOGIT_HS,'hs'),
 'HS_Interactions':(LOGIT_INTER,'interactions'),
 'HGB_Base':(HGB_BASE,'base'),
 'HGB_HS':(HGB_HS,'hs'),
}
print('Training count',int(TRAIN.sum()),'positive labels',int(n_pos))

# Predict in chunks, only for 2022 calibration + 2023-2026 development.
SCORE={}
mask_pred=CAL|DEV
indices=np.flatnonzero(mask_pred)
for name,(model,which) in MODELS.items():
    scores=np.full(len(T),np.nan,dtype=np.float64)
    for block in np.array_split(indices,max(1,int(np.ceil(len(indices)/180000)))):
        xx= X_INTERACT[block] if which=='interactions' else (X[block,:2] if which=='base' else X[block])
        scores[block]=model.predict_proba(xx)[:,1]
    assert np.isfinite(scores[mask_pred]).all(),f'{name}: nonfinite scores'
    SCORE[name]=scores
    print(name,'score computed on',len(indices),'2022+development minutes',flush=True)
# Free the largest temporary arrays; the score arrays and base inputs remain.
del X_INTERACT,zv,ts,hs

PASS: frozen M0 logistic parameters
Training count 524087 positive labels 18779
Base_Logit score computed on 2276212 2022+development minutes
HS_Linear score computed on 2276212 2022+development minutes
HS_Interactions score computed on 2276212 2022+development minutes
HGB_Base score computed on 2276212 2022+development minutes
HGB_HS score computed on 2276212 2022+development minutes


## 3. Frozen scheduler and threshold calibration

The calendar scheduler takes the first eligible minute, with 20 alerts per UTC calendar month, 240 per year and a 240-minute cooldown. The threshold grid uses 2022 scores only, not outcomes, and I keep the frozen `score >= cutoff` rule.

In [4]:
@njit(cache=False)
def calendar_core(t,elig,years,months,B,cap,cooldown_ns):
    out=np.empty(len(t),dtype=np.int64)
    n=0; month_count=0; year_count=0; last_m=-1; last_y=-1; last=-(2**62)
    for i in range(len(t)):
        if not elig[i]: continue
        yr=years[i]; mo=months[i]; ti=t[i]
        if yr!=last_y: year_count=0; last_y=yr
        if mo!=last_m: month_count=0; last_m=mo
        if month_count>=cap or year_count>=B or ti-last<cooldown_ns: continue
        out[n]=ti;n+=1;month_count+=1;year_count+=1;last=ti
    return out[:n].copy()

def scheduler(time_ns,score,threshold):
    tm=pd.to_datetime(time_ns,utc=True)
    yrs=np.asarray(tm.year,dtype=np.int64)
    months=np.asarray(tm.year*12+tm.month,dtype=np.int64)
    e=np.asarray(np.isfinite(score)&(score>=threshold),dtype=np.bool_)
    return calendar_core(np.asarray(time_ns,np.int64),e,yrs,months,BUDGET_YEAR,BUDGET_MONTH,COOLDOWN*NS)

def score_grid(sc):
    Q=np.unique(np.r_[0.,np.linspace(.10,.995,33),.997,.999,.9999,1.])
    return np.unique(np.r_[np.nextafter(sc.min(),-np.inf),np.quantile(sc,Q),np.nextafter(sc.max(),np.inf)])

def calibrate(time_ns,sc):
    rows=[]
    for th in score_grid(sc):
        count=len(scheduler(time_ns,sc,th))
        rows.append((float(th),count,int(np.count_nonzero(sc>=th))))
    d=pd.DataFrame(rows,columns=['threshold','cal_alerts','raw_candidates'])
    row=d[d.cal_alerts==d.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(row.threshold),int(row.cal_alerts),d

T_CAL=T[CAL];T_DEV=T[DEV]
THR={}; ALARMS={}; CAL_COUNTS={}
for name in MODELS:
    thr,ncal,_=calibrate(T_CAL,SCORE[name][CAL])
    THR[name]=thr;CAL_COUNTS[name]=ncal
    ALARMS[name]=scheduler(T_DEV,SCORE[name][DEV],thr)
    print(f'{name:14s} 2022 cal alerts={ncal:3d}  threshold={thr:.6g}  development alerts={len(ALARMS[name])}')
assert all(len(x)<=40*20 for x in ALARMS.values()),'Exceeded maximum 40-month 20-per-month budget'

Base_Logit     2022 cal alerts=240  threshold=0.669899  development alerts=800
HS_Linear      2022 cal alerts=240  threshold=0.671414  development alerts=800
HS_Interactions 2022 cal alerts=240  threshold=0.667268  development alerts=800
HGB_Base       2022 cal alerts=240  threshold=0.621315  development alerts=800
HGB_HS         2022 cal alerts=240  threshold=0.626659  development alerts=800


## 4. Reproduction of the notebook 47 breach- and onset-anchored event outcomes

With the notebook 47 HGB pair I reproduce the notebook 47–49 event and alert counts exactly.

In [5]:
def event_catches(events, alerts, anchor='breach'):
    alert=np.asarray(alerts,dtype=np.int64)
    rows=[]
    for e in events.itertuples(index=False):
        on=e.onset_1pct_utc.value;b=e.breach_utc.value
        anchor_ns=on if anchor=='onset' else b
        lo=anchor_ns-LEAD_MAX*NS; hi=anchor_ns-LEAD_MIN*NS
        l=np.searchsorted(alert,lo,side='left')
        r=np.searchsorted(alert,hi,side='right')
        eligible=alert[l:r]
        good=eligible[eligible<on]
        rows.append((e.event_id,len(good)>0,(on-int(good[0]))/NS if len(good) else np.nan))
    return pd.DataFrame(rows,columns=['event_id','hit','lead_to_onset_minutes'])

rows=[];HITS={}
for h in (240,1440):
    for name in MODELS:
        for anchor in ('breach','onset'):
            E = EVAL_ON[h] if anchor=='onset' else EVAL[h]
            hit=event_catches(E,ALARMS[name],anchor)
            HITS[(h,name,anchor)]=hit
            n=len(ALARMS[name]);k=int(hit.hit.sum())
            rows.append({'H':h,'model':name,'anchor':anchor,'events':len(E),
                         'alerts':n,'pre_catches':k,'pre_recall':k/len(E),
                         'pre_catches_per_100_alerts':100*k/n if n else np.nan,
                         'median_lead_minutes':hit.lead_to_onset_minutes.median(),
                         '2022_calibration_alerts':CAL_COUNTS[name]})
RESULTS=pd.DataFrame(rows)
from IPython.display import display
display(RESULTS)

,H,model,anchor,events,alerts,pre_catches,pre_recall,pre_catches_per_100_alerts,median_lead_minutes,2022_calibration_alerts
0,240,Base_Logit,breach,83,800,22,0.265060,2.750,25.0,240
1,240,Base_Logit,onset,83,800,15,0.180723,1.875,72.0,240
2,240,HS_Linear,breach,83,800,18,0.216867,2.250,42.5,240
3,240,HS_Linear,onset,83,800,13,0.156627,1.625,64.0,240
4,240,HS_Interactions,breach,83,800,14,0.168675,1.750,60.5,240
5,240,HS_Interactions,onset,83,800,14,0.168675,1.750,94.0,240
6,240,HGB_Base,breach,83,800,13,0.156627,1.625,27.0,240
7,240,HGB_Base,onset,83,800,9,0.108434,1.125,97.0,240
8,240,HGB_HS,breach,83,800,12,0.144578,1.500,65.5,240
9,240,HGB_HS,onset,83,800,16,0.192771,2.000,134.5,240


In [6]:
CHECK = {
 'onset_HGB_Base': int(RESULTS.query('H==240 and model=="HGB_Base" and anchor=="onset"').pre_catches.iloc[0]),
 'onset_HGB_HS': int(RESULTS.query('H==240 and model=="HGB_HS" and anchor=="onset"').pre_catches.iloc[0]),
 'breach_HGB_Base': int(RESULTS.query('H==240 and model=="HGB_Base" and anchor=="breach"').pre_catches.iloc[0]),
 'breach_HGB_HS': int(RESULTS.query('H==240 and model=="HGB_HS" and anchor=="breach"').pre_catches.iloc[0]),
 'base_alerts':len(ALARMS['HGB_Base']),'hs_alerts':len(ALARMS['HGB_HS'])
}
assert CHECK == {'onset_HGB_Base':9,'onset_HGB_HS':16,'breach_HGB_Base':13,
                'breach_HGB_HS':12,'base_alerts':800,'hs_alerts':800}, CHECK
assert abs(THR['HGB_Base']-0.6213147944479045)<1e-9
assert abs(THR['HGB_HS']-0.6266591400580771)<1e-9
print('PASS notebook 47/48/49 reproduction',CHECK)

PASS notebook 47/48/49 reproduction {'onset_HGB_Base': 9, 'onset_HGB_HS': 16, 'breach_HGB_Base': 13, 'breach_HGB_HS': 12, 'base_alerts': 800, 'hs_alerts': 800}


## 5. The eight policies, declared before the development evaluation

The policies transform the already trained HGB+Spectral scores without retuning the model: the tie-level rule uses the 2021 training median of `z_hs`, the tie-rising rule only the past 15-minute change in `z_hs`, and score jitter is `score + 0.002 × clip(z_hs, -3, 3)`, with a cutoff that I calibrate only on unlabeled 2022 data. The 10+10 monthly quota prevents a policy from using all 20 alerts before the second half of the month and may give fewer than 800 alerts in total.

In [7]:
@njit(cache=False)
def scheduler_with_halfmonth_core(t, eligible, yrs, mons, halfs, cap_year, cap_month, cap_half, cd_ns):
    out=np.empty(len(t),dtype=np.int64); n=0
    yprev=-1; mprev=-1; hprev=-1; ny=0; nm=0; nh=0; last=-(2**62)
    for i in range(len(t)):
        if not eligible[i]:
            continue
        y=int(yrs[i]);m=int(mons[i]);h=int(halfs[i]);ti=int(t[i])
        if y != yprev:
            ny=0; yprev=y
        if m != mprev:
            nm=0;mprev=m
        if h != hprev:
            nh=0;hprev=h
        if ny>=cap_year or nm>=cap_month or nh>=cap_half or ti-last < cd_ns:
            continue
        out[n]=ti;n+=1;ny+=1;nm+=1;nh+=1;last=ti
    return out[:n].copy()

def flexible_scheduler(times_ns, scores, threshold, policy='standard', spectral=None,
                       spectral_delta=None, halfmonth=False, spectral_median=None):
    """The same UTC no-lookahead decision rule applied on any contiguous minute subset."""
    t=np.asarray(times_ns,dtype=np.int64)
    sc=np.asarray(scores,dtype=np.float64)
    assert len(t)==len(sc) and np.all(np.diff(t)>0)
    tm=pd.to_datetime(t,utc=True)
    yrs=np.asarray(tm.year,dtype=np.int64)
    mons=np.asarray(tm.year*12+tm.month,dtype=np.int64)
    if halfmonth:
        half_id=np.asarray(mons*2+(np.asarray(tm.day)>15),dtype=np.int64)
        cap_half=10
    else:
        half_id=np.asarray(mons*2+(np.asarray(tm.day)>15),dtype=np.int64)
        cap_half=BUDGET_MONTH
    finite=np.isfinite(sc)
    eq=finite & (sc==threshold)
    gt=finite & (sc>threshold)
    if policy=='standard':
        eligible=gt|eq
    elif policy=='strict':
        eligible=gt
    elif policy=='tie_level':
        assert spectral is not None and spectral_median is not None
        ss=np.asarray(spectral,dtype=float)
        eligible=gt | (eq & np.isfinite(ss) & (ss>spectral_median))
    elif policy=='tie_rise15':
        assert spectral_delta is not None
        delta=np.asarray(spectral_delta,dtype=float)
        eligible=gt | (eq & np.isfinite(delta) & (delta>0))
    else:
        raise ValueError(policy)
    if not halfmonth:
        # Baseline scheduler path, reproduced exactly.
        return calendar_core(t,np.asarray(eligible,dtype=np.bool_),yrs,mons,
                             BUDGET_YEAR,BUDGET_MONTH,COOLDOWN*NS)
    return scheduler_with_halfmonth_core(t,np.asarray(eligible,dtype=np.bool_),yrs,mons,
                                        half_id,BUDGET_YEAR,BUDGET_MONTH,cap_half,COOLDOWN*NS)

S50={
    'notebook':'50','status':'exploratory_posthoc_model_policy_development',
    'source':'frozen BTC Spot inputs, original 47 fitted HGB models',
    'train':'2021','unlabeled_threshold_calibration':'2022',
    'dev_evaluation':'2023-01 through 2026-04',
    'primary':'4h Breach-anchor pre-onset catches per 100 alerts',
    'selected_secondary':'4h Onset-anchor pre-onset catches per 100 alerts',
    'monthly_cap':20,'annual_cap':240,'cooldown_minutes':240,
    'jitter_coefficient':0.002,'jitter_clip_zhs':[-3,3],
    'halfmonth_budget':[10,10],'spectral_trend_lag_minutes':15,
    'spectral_level_reference':'2021 training median',
    'arms': ['BASE','HS_ORIGINAL','HS_STRICT','HS_TIE_LEVEL','HS_TIE_RISE15',
             'HS_JITTER_LEVEL','HS_HALFMONTH','HS_JITTER_HALFMONTH'],
    'no_tuning_or_selection_on_development_outcomes': True,
    'not_independent_validation':True,
    'registered_prospective_period_not_accessed':'2026-12 through 2027-11',
    'negative_47_primary_result_must_be_reported': True
}
(OUT/'50_SPEC.json').write_text(json.dumps(S50,indent=2,ensure_ascii=False))

# Unit tests for the calendar and eligibility rules before touching development outcomes.
def unit_policy_tests():
    idx=pd.date_range('2022-01-01', periods=600, freq='min',tz='UTC').asi8
    sc=np.ones(len(idx))*0.65
    sc[100:110]=0.7
    ss=np.linspace(-1,1,len(idx))
    x=flexible_scheduler(idx,sc,0.65,'standard',spectral=ss)
    y=flexible_scheduler(idx,sc,0.65,'strict',spectral=ss)
    assert len(x)>0 and np.all(np.diff(x)>=COOLDOWN*NS)
    assert len(y)>0 and np.all((y>=idx[100]) & (y<=idx[109]))
    a=flexible_scheduler(idx,np.full(len(idx),0.65),0.65,'tie_level',spectral=ss,spectral_median=0.0)
    assert len(a)>0 and a[0]>=idx[300]
    b=flexible_scheduler(idx,sc,0.65,'standard',spectral=ss,halfmonth=True)
    assert len(b)>0 and np.all(np.diff(b)>=COOLDOWN*NS)
    # Test half-month split: 10 alerts in the first half and 10 in the second.
    ind=pd.date_range('2022-02-01',periods=28*24,freq='h',tz='UTC').asi8
    h=flexible_scheduler(ind,np.ones(len(ind)),0.5,'standard',halfmonth=True)
    hd=pd.to_datetime(h,utc=True).day
    assert int((hd<=15).sum())<=10 and int((hd>15).sum())<=10 and len(h)<=20
unit_policy_tests()
print('PASS scheduler synthetic tests and specification written')

PASS scheduler synthetic tests and specification written


## 6. Causally available spectral signals and policy calibration on 2022 only

`HS_TIE_LEVEL` and `HS_TIE_RISE15` change only whether a minute exactly on the HGB cutoff plateau can fire, while I choose the cutoffs of `HS_JITTER_LEVEL`, `HS_HALFMONTH` and `HS_JITTER_HALFMONTH` from 2022 scores without any 2023–26 labels. I save thresholds, calendar budgets and observed counts and keep the frozen 2022-calibrated first-eligible scheduler as the benchmark.

In [8]:
# The notebook 47 and 49 score arrays are already available: SCORE['HGB_Base'], SCORE['HGB_HS'].
# All scoring uses current and past signals, with a 15-minute shift on the complete 1-minute grid.
train_median=float(np.median(X[TRAIN,2]))
hs_val=np.asarray(X[:,2],dtype=float)
hs_delta=np.full(len(T),np.nan,dtype=np.float64)
hs_delta[15:]=hs_val[15:]-hs_val[:-15]
JITTER_COEF=0.002
jitter=np.zeros(len(T),dtype=float)
mask_both=CAL|DEV
jitter[mask_both]=JITTER_COEF*np.clip(hs_val[mask_both],-3.0,3.0)
score_hs_jitter=np.full(len(T),np.nan,dtype=np.float64)
score_hs_jitter[mask_both]=SCORE['HGB_HS'][mask_both]+jitter[mask_both]
assert np.isfinite(score_hs_jitter[mask_both]).all()

POLICY_META={
 'BASE':('HGB_Base','standard',False,False),
 'HS_ORIGINAL':('HGB_HS','standard',False,False),
 'HS_STRICT':('HGB_HS','strict',False,False),
 'HS_TIE_LEVEL':('HGB_HS','tie_level',False,False),
 'HS_TIE_RISE15':('HGB_HS','tie_rise15',False,False),
 'HS_JITTER_LEVEL':('HGB_HS','standard',False,True),
 'HS_HALFMONTH':('HGB_HS','standard',True,False),
 'HS_JITTER_HALFMONTH':('HGB_HS','standard',True,True),
}

def score_for_policy(arm):
    return score_hs_jitter if POLICY_META[arm][-1] else SCORE[POLICY_META[arm][0]]

def dispatch_policy(arm,mask,threshold):
    model,tie_rule,halfmonth,is_jitter=POLICY_META[arm]
    sc=score_for_policy(arm)[mask]
    ts=T[mask]
    if not len(ts):
        return np.empty(0,dtype=np.int64)
    return flexible_scheduler(ts,sc,float(threshold),policy=tie_rule,
          spectral=hs_val[mask],spectral_delta=hs_delta[mask],
          halfmonth=halfmonth,spectral_median=train_median)

# Baseline threshold for score/tie variants; 2022-only calibration for the three modified
# score/budget policies. The policy set was frozen in 50_SPEC.json before any DEV outcomes.
CALIBRATION=[]; P50={};TH50={};N50={}
for arm in S50['arms']:
    if arm in ('BASE','HS_ORIGINAL','HS_STRICT','HS_TIE_LEVEL','HS_TIE_RISE15'):
        # Keep the baseline calibrated score cutoff; only candidate eligibility varies.
        th=THR[POLICY_META[arm][0]]
        cgrid=None
    else:
        # 2022 outcome-free calibration. First maximize attainable alert count, then choose
        # the most selective threshold among tied budget-saturating candidates.
        sc=score_for_policy(arm)[CAL]
        qgrid=score_grid(sc)
        trials=[]
        for th0 in qgrid:
            n=len(dispatch_policy(arm,CAL,float(th0)))
            nc=int(np.count_nonzero(sc>=th0))
            trials.append((float(th0),int(n),nc))
        cgrid=pd.DataFrame(trials,columns=['threshold','cal_alerts','raw_candidates'])
        # Use the exact same eligibility-maximizing, selectivity tie rule as notebook 47.
        best=cgrid[cgrid.cal_alerts==cgrid.cal_alerts.max()].sort_values(['raw_candidates','threshold'],ascending=[True,False]).iloc[0]
        th=float(best.threshold)
        cgrid.assign(policy=arm).to_csv(OUT/f'50_calibration_grid_{arm}.csv',index=False)
    TH50[arm]=float(th)
    cal_alerts=dispatch_policy(arm,CAL,th)
    dev_alerts=dispatch_policy(arm,DEV,th)
    P50[arm]=dev_alerts;N50[arm]=len(dev_alerts)
    CALIBRATION.append({'arm':arm,'threshold':th,'2022_alerts':len(cal_alerts),
      'dev_alerts':len(dev_alerts),'equal_2022_budget':len(cal_alerts)==240,
      'equal_dev_budget':len(dev_alerts)==800,'halfmonth':POLICY_META[arm][2],
      'jitter':POLICY_META[arm][3], 'score_variant':POLICY_META[arm][0],
      'tie_rule':POLICY_META[arm][1]})
CALIBRATION=pd.DataFrame(CALIBRATION)
assert np.array_equal(P50['BASE'],ALARMS['HGB_Base'])
assert np.array_equal(P50['HS_ORIGINAL'],ALARMS['HGB_HS'])
assert all(0<=int(n)<=800 for n in N50.values())
CALIBRATION.to_csv(OUT/'50_1_policy_calibration.csv',index=False)
from IPython.display import display
display(CALIBRATION)

,arm,threshold,2022_alerts,dev_alerts,equal_2022_budget,equal_dev_budget,halfmonth,jitter,score_variant,tie_rule
0,BASE,0.621315,240,800,True,True,False,False,HGB_Base,standard
1,HS_ORIGINAL,0.626659,240,800,True,True,False,False,HGB_HS,standard
2,HS_STRICT,0.626659,240,800,True,True,False,False,HGB_HS,strict
3,HS_TIE_LEVEL,0.626659,240,800,True,True,False,False,HGB_HS,tie_level
4,HS_TIE_RISE15,0.626659,240,800,True,True,False,False,HGB_HS,tie_rise15
5,HS_JITTER_LEVEL,0.626958,240,800,True,True,False,True,HGB_HS,standard
6,HS_HALFMONTH,0.626659,240,800,True,True,True,False,HGB_HS,standard
7,HS_JITTER_HALFMONTH,0.626958,240,800,True,True,True,True,HGB_HS,standard


## 7. Evaluation of all policies, breach-anchored endpoint first

Four-hour breach-anchored catches per 100 alerts are the primary comparison endpoint and 4-hour onset-anchored catches a post hoc secondary endpoint; I also report 24-hour breach- and onset-anchored results, and I choose or tune no policy on these results. The paired event bootstrap is descriptive only and does not remove time dependence, repeated testing or outcome selection.

In [9]:
REPORT=[]; MATRIX=[]
for h in (240,1440):
    for anch in ('breach','onset'):
        ev=EVAL[h] if anch=='breach' else EVAL_ON[h]
        for arm in S50['arms']:
            hc=event_catches(ev,P50[arm],anch)
            n=len(P50[arm]);k=int(hc.hit.sum())
            REPORT.append({'H':h,'anchor':anch,'arm':arm,'events':len(ev),
                'alerts':n,'pre_catches':k,'catches_per_100_alerts':100*k/n if n else np.nan,
                'pre_recall':k/len(ev) if len(ev) else np.nan,
                'median_lead_min':hc.lead_to_onset_minutes.median(),
                'equal_dev_budget':n==800, 'label':'exploratory_posthoc'})
            for row in hc.itertuples(index=False):
                MATRIX.append({'H':h,'anchor':anch,'arm':arm,'event_id':row.event_id,'hit':bool(row.hit),'lead_min':row.lead_to_onset_minutes})
REPORT=pd.DataFrame(REPORT)
MATRIX=pd.DataFrame(MATRIX)
REPORT.to_csv(OUT/'50_2_all_policy_results.csv',index=False)
MATRIX.to_csv(OUT/'50_3_event_level_all_policies.csv',index=False)

# All event comparisons must be paired on the same catalog. Compare to both baseline HGB arms.
BOOT=[];rng=np.random.default_rng(20261150)
for h in (240,1440):
    for anch in ('breach','onset'):
        a=MATRIX[(MATRIX.H==h)&(MATRIX.anchor==anch)]
        p=a.pivot(index='event_id',columns='arm',values='hit')
        assert list(p.columns)==sorted(S50['arms']) and not p.isna().any().any()
        for ref in ('BASE','HS_ORIGINAL'):
            r=p[ref].to_numpy(bool)
            for arm in S50['arms']:
                if arm==ref: continue
                z=p[arm].to_numpy(bool)
                delta=z.astype(int)-r.astype(int)
                draws=np.asarray([delta[rng.integers(0,len(delta),len(delta))].sum() for _ in range(2000)])
                BOOT.append({'H':h,'anchor':anch,'reference':ref,'candidate':arm,
                    'reference_hits':int(r.sum()),'candidate_hits':int(z.sum()),
                    'delta_hits':int(delta.sum()),'candidate_only':int((z&~r).sum()),
                    'reference_only':int((r&~z).sum()),
                    'naive_event_boot_lo':float(np.quantile(draws,.025)),
                    'naive_event_boot_hi':float(np.quantile(draws,.975)),
                    'posthoc_exploratory_only':True})
BOOT=pd.DataFrame(BOOT)
BOOT.to_csv(OUT/'50_4_paired_event_descriptive.csv',index=False)
print('Primary: 4h breach-anchored pre-onset catches')
display(REPORT[(REPORT.H==240)&(REPORT.anchor=='breach')])
print('Selected-after-viewing secondary: 4h Onset-anchored pre-onset catches')
display(REPORT[(REPORT.H==240)&(REPORT.anchor=='onset')])

Primary: 4h breach-anchored pre-onset catches


,H,anchor,arm,events,alerts,pre_catches,catches_per_100_alerts,pre_recall,median_lead_min,equal_dev_budget,label
0,240,breach,BASE,83,800,13,1.625,0.156627,27.0,True,exploratory_posthoc
1,240,breach,HS_ORIGINAL,83,800,12,1.500,0.144578,65.5,True,exploratory_posthoc
2,240,breach,HS_STRICT,83,800,15,1.875,0.180723,38.0,True,exploratory_posthoc
3,240,breach,HS_TIE_LEVEL,83,800,12,1.500,0.144578,65.0,True,exploratory_posthoc
4,240,breach,HS_TIE_RISE15,83,800,14,1.750,0.168675,45.5,True,exploratory_posthoc
5,240,breach,HS_JITTER_LEVEL,83,800,14,1.750,0.168675,45.5,True,exploratory_posthoc
6,240,breach,HS_HALFMONTH,83,800,7,0.875,0.084337,79.0,True,exploratory_posthoc
7,240,breach,HS_JITTER_HALFMONTH,83,800,9,1.125,0.108434,53.0,True,exploratory_posthoc


Selected-after-viewing secondary: 4h Onset-anchored pre-onset catches


,H,anchor,arm,events,alerts,pre_catches,catches_per_100_alerts,pre_recall,median_lead_min,equal_dev_budget,label
8,240,onset,BASE,83,800,9,1.125,0.108434,97.0,True,exploratory_posthoc
9,240,onset,HS_ORIGINAL,83,800,16,2.000,0.192771,134.5,True,exploratory_posthoc
10,240,onset,HS_STRICT,83,800,15,1.875,0.180723,129.0,True,exploratory_posthoc
11,240,onset,HS_TIE_LEVEL,83,800,16,2.000,0.192771,134.5,True,exploratory_posthoc
12,240,onset,HS_TIE_RISE15,83,800,16,2.000,0.192771,139.0,True,exploratory_posthoc
13,240,onset,HS_JITTER_LEVEL,83,800,16,2.000,0.192771,139.0,True,exploratory_posthoc
14,240,onset,HS_HALFMONTH,83,800,14,1.750,0.168675,129.5,True,exploratory_posthoc
15,240,onset,HS_JITTER_HALFMONTH,83,800,13,1.625,0.156627,90.0,True,exploratory_posthoc


## 8. Alert timing shifts versus cutoff artifacts

For each policy I compute exact timestamp overlap and nearest-alert distances against HGB+Spectral and track the eight rescued onset events (the `new_only` events of notebooks 47 and 48), reporting both hit gains and losses.

In [10]:
# One-to-one alert timestamp pairing within a fixed tolerance (chronological greedy matching).
# This is descriptive and intentionally not an optimal transport algorithm.
def match_tolerance(a,b,minutes):
    aa=np.asarray(a,np.int64);bb=np.asarray(b,np.int64)
    i=j=matched=0; tol=minutes*NS
    while i<len(aa) and j<len(bb):
        dt=int(aa[i])-int(bb[j])
        if abs(dt)<=tol:
            matched+=1;i+=1;j+=1
        elif dt<0:
            i+=1
        else:
            j+=1
    return matched
OVERLAP=[]
for arm in S50['arms']:
    aa=P50['HS_ORIGINAL'];bb=P50[arm]
    for tol in (0,5,15,30,60):
        OVERLAP.append({'arm':arm,'tolerance_minutes':tol,
            'reference_alerts':len(aa),'candidate_alerts':len(bb),
            'paired_alerts':match_tolerance(aa,bb,tol),
            'percentage_reference_matched':100*match_tolerance(aa,bb,tol)/len(aa)})
OVERLAP=pd.DataFrame(OVERLAP)
OVERLAP.to_csv(OUT/'50_5_policy_alert_overlap.csv',index=False)

# The eight spectral-only onset events selected in notebooks 47/48; fixed identifiers purely for
# forensic reporting, never for fitting, threshold selection or policy decisions.
ORIGINAL_EIGHT=(190,196,204,229,237,246,249,263)
base_hits=MATRIX.query('H==240 and anchor=="onset" and arm=="BASE"').set_index('event_id').hit
hs_hits=MATRIX.query('H==240 and anchor=="onset" and arm=="HS_ORIGINAL"').set_index('event_id').hit
assert set(base_hits.index[hs_hits & ~base_hits].tolist())==set(ORIGINAL_EIGHT)
EIGHT=[]
for arm in S50['arms']:
    hit=MATRIX.query('H==240 and anchor=="onset" and arm==@arm').set_index('event_id').hit
    for e in ORIGINAL_EIGHT:
        EIGHT.append({'arm':arm,'event_id':e,'original_spectral_hit':bool(hs_hits.loc[e]),
            'candidate_hit':bool(hit.loc[e]),'status':'maintained' if bool(hit.loc[e]) else 'lost'})
EIGHT=pd.DataFrame(EIGHT)
EIGHT.to_csv(OUT/'50_6_eight_rescues_by_policy.csv',index=False)
print('Retention of the eight new-only rescues per arm (descriptive, not a selection metric):')
display(EIGHT.groupby('arm').candidate_hit.agg(['sum','count']).reset_index())

Retention of the eight new-only rescues per arm (descriptive, not a selection metric):


,arm,sum,count
0,BASE,0,8
1,HS_HALFMONTH,5,8
2,HS_JITTER_HALFMONTH,5,8
3,HS_JITTER_LEVEL,7,8
4,HS_ORIGINAL,8,8
5,HS_STRICT,6,8
6,HS_TIE_LEVEL,8,8
7,HS_TIE_RISE15,7,8


## 9. Score-plateau and monthly-cap diagnostics

For each policy I report the share of alerts on the frozen score threshold and the speed of monthly budget use. Half-month policies reserve 10 of the 20 monthly slots for day 16 onward and may lose some raw alert yield.

In [11]:
PDIAG=[];MDIAG=[]
full_devt=pd.to_datetime(T_DEV,utc=True)
orig_hs=SCORE['HGB_HS'][DEV]
for arm in S50['arms']:
    a=P50[arm]; dt=pd.to_datetime(a,utc=True)
    inds=np.searchsorted(T_DEV,a)
    assert np.array_equal(T_DEV[inds],a)
    n=len(a)
    PDIAG.append({'arm':arm,'alerts':n,
        'original_score_cutoff_equal_alerts':int(np.isclose(orig_hs[inds],THR['HGB_HS'],atol=0,rtol=0).sum()),
        'median_alert_day_of_month':float(np.median(dt.day)) if n else np.nan,
        'day_1to15_alerts':int(np.sum(dt.day<=15)),'day_16plus_alerts':int(np.sum(dt.day>15))})
    for month in pd.period_range('2023-01','2026-04',freq='M'):
        start=pd.Timestamp(month.start_time,tz='UTC').value
        stop=pd.Timestamp((month+1).start_time,tz='UTC').value
        z=a[(a>=start)&(a<stop)]
        d=pd.to_datetime(z,utc=True).day
        MDIAG.append({'arm':arm,'month':str(month),'alerts':len(z),
            'first_half':int((d<=15).sum()),'second_half':int((d>15).sum()),
            'first_alert_utc':str(pd.Timestamp(int(z[0]),tz='UTC')) if len(z) else '',
            'last_alert_utc':str(pd.Timestamp(int(z[-1]),tz='UTC')) if len(z) else ''})
PDIAG=pd.DataFrame(PDIAG);MDIAG=pd.DataFrame(MDIAG)
PDIAG.to_csv(OUT/'50_7_score_tie_and_timing_diagnostics.csv',index=False)
MDIAG.to_csv(OUT/'50_8_monthly_budget_consumption.csv',index=False)
assert MDIAG.alerts.max()<=20
assert MDIAG.groupby('arm').alerts.sum().to_dict()==N50
for arm in ('HS_HALFMONTH','HS_JITTER_HALFMONTH'):
    assert (MDIAG.loc[MDIAG.arm==arm,'first_half']<=10).all()
    assert (MDIAG.loc[MDIAG.arm==arm,'second_half']<=10).all()
display(PDIAG)

,arm,alerts,original_score_cutoff_equal_alerts,median_alert_day_of_month,day_1to15_alerts,day_16plus_alerts
0,BASE,800,0,5.0,792,8
1,HS_ORIGINAL,800,128,4.0,800,0
2,HS_STRICT,800,0,5.0,800,0
3,HS_TIE_LEVEL,800,103,4.0,800,0
4,HS_TIE_RISE15,800,44,5.0,800,0
5,HS_JITTER_LEVEL,800,0,5.0,800,0
6,HS_HALFMONTH,800,126,12.5,400,400
7,HS_JITTER_HALFMONTH,800,0,12.5,400,400


## 10. Descriptive yearly stability (not a new holdout)

I have inspected every development year in earlier notebooks, so this breakdown can show whether gains concentrate in one period but cannot validate a selected policy.

In [12]:
YR=[]
for yr in (2023,2024,2025,2026):
    # Pre-onset outcomes (breach and onset anchors) for every event whose onset is in this year.
    for anch in ('breach','onset'):
        ee=EVAL[240] if anch=='breach' else EVAL_ON[240]
        stamp='breach_utc' if anch=='breach' else 'onset_1pct_utc'
        eyr=ee[pd.to_datetime(ee[stamp],utc=True).dt.year==yr]
        for arm in S50['arms']:
            hit=event_catches(eyr,P50[arm],anch) if len(eyr) else pd.DataFrame({'hit':[]})
            num_alerts=int(np.sum(pd.to_datetime(P50[arm],utc=True).year==yr))
            YR.append({'year':yr,'anchor':anch,'arm':arm,'events':len(eyr),
                'year_alerts':num_alerts,'hits':int(hit.hit.sum()),
                'per_100_year_alerts':100*hit.hit.sum()/num_alerts if num_alerts else np.nan})
YEARLY=pd.DataFrame(YR)
YEARLY.to_csv(OUT/'50_9_development_years_DESCRIPTIVE.csv',index=False)

## 11. Interpretation

The notebook 47 and 49 result is unchanged: the spectral HGB fails the primary breach-anchored improvement screen. I read a policy arm that looks better on these already seen data as a hypothesis about threshold and tie handling or alert capacity allocation, not as a confirmed forecasting improvement.

In [13]:
# Save a portable, minimal JSON summary; archive all policies and the exact frozen specification.
PRIMARY=REPORT[(REPORT.H==240)&(REPORT.anchor=='breach')].to_dict('records')
SECONDARY=REPORT[(REPORT.H==240)&(REPORT.anchor=='onset')].to_dict('records')
summary={
  'notebook':'50','status':'EXPLORATORY_POSTHOC_DEVELOPMENT_NOT_VALIDATION',
  'original_47_reproduction':CHECK,
  'original_47_primary_screen':'FAILED',
  'candidate_policy_count':len(S50['arms']),
  'primary_breach_four_hour':PRIMARY,
  'secondary_onset_four_hour_selected_after_inspection':SECONDARY,
  'calibration':CALIBRATION.to_dict('records'),
  'original_eight_event_fates':EIGHT.groupby('arm').candidate_hit.sum().to_dict(),
  'no_new_holdout_tested':True,
  'no_development_winner_selected':True,
  'no_model_retuning_on_2023_2026_labels':True,
  'original_registration_unchanged':True,
  'next_step_if_promising':'Freeze one policy from 2021/2022 criteria and evaluate on a genuinely untouched window outside existing inspections and preregistration.'
}
(OUT/'50_SUMMARY.json').write_text(json.dumps(summary,indent=2,default=str))
print('DONE. Results saved to:',OUT)
print('The primary breach-anchored result stays negative; all variants are reported, with no claim of independent validation.')

DONE. Results saved to: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/spectral_tie_timing_policy_development_v1
The primary breach-anchored result stays negative; all variants are reported, with no claim of independent validation.
